# Analisis de Resenas - Wise.com (Trustpilot)

**Autor:** [Tu nombre]
**Empresa analizada:** Wise.com (transferencias internacionales)
**Sector:** Money & Insurance
**Dataset:** Trustpilot Reviews 123k (100 resenas por empresa, 20 por estrella)

---

### Objetivos del proyecto
1. Determinar si la mayoria de las resenas de Wise son positivas o negativas, y comparar con la competencia.
2. Identificar los temas principales que tratan las resenas (topic modeling).
3. Cruzar sentimiento con topics para saber en que temas Wise destaca o falla.
4. Proponer areas de mejora basadas en datos.

### Herramientas utilizadas
- **Sentimiento:** `nlptown/bert-base-multilingual-uncased-sentiment` (BERT, 1-5 estrellas)
- **Topics:** BERTopic (embeddings + UMAP + HDBSCAN + c-TF-IDF)
- **Visualizacion:** matplotlib, seaborn, wordcloud


---
## Paso 1: Carga de librerias y configuracion

Importamos todas las librerias necesarias y definimos las constantes del proyecto:
- `CSV_PATH`: ruta al dataset de Trustpilot.
- `TARGET`: empresa objetivo (wise.com).
- `CATEGORY`: sector para la comparativa (Money & Insurance).
- `stars_to_label()`: funcion que mapea las estrellas predichas por el modelo a etiquetas de sentimiento:
  - 1-2 estrellas = **Negativo**
  - 3 estrellas = **Neutro**
  - 4-5 estrellas = **Positivo**


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

# Configuracion
CSV_PATH = 'trustpilot-reviews-123k.csv'
TARGET   = 'wise.com'
CATEGORY = 'Money & Insurance'

def stars_to_label(s):
    """Convierte prediccion de estrellas (1-5) a etiqueta de sentimiento."""
    if s <= 2: return 'Negativo'
    if s == 3: return 'Neutro'
    return 'Positivo'

PALETTE = {'Positivo': '#2ecc71', 'Neutro': '#f39c12', 'Negativo': '#e74c3c'}
print('Librerias cargadas OK')


---
## Paso 2: Carga y exploracion del dataset

Cargamos el CSV completo (123k resenas) y filtramos:
- **df_sector**: todas las resenas del sector `Money & Insurance` (~5.729 resenas, 70 empresas).
- **df_wise**: solo las 100 resenas de `wise.com`.

El dataset esta perfectamente balanceado: cada empresa tiene exactamente 100 resenas,
con 20 resenas por cada nivel de estrellas (1 a 5).

Usamos `reset_index(drop=True)` para evitar problemas de alineacion de indices
al asignar nuevas columnas mas adelante.


In [ ]:
df_all = pd.read_csv(CSV_PATH)
print(f'Dataset completo: {df_all.shape[0]} resenas, {df_all.shape[1]} columnas')
print(f'Columnas: {df_all.columns.tolist()}')
df_all.head(3)


In [ ]:
# Filtrar sector y empresa objetivo
df_sector = df_all[df_all['category'] == CATEGORY].reset_index(drop=True).copy()
df_wise   = df_all[df_all['company'] == TARGET].reset_index(drop=True).copy()

n_emp = df_sector['company'].nunique()
print(f'Sector "{CATEGORY}": {len(df_sector)} resenas | {n_emp} empresas')
print(f'Wise: {len(df_wise)} resenas')
print()
print('Distribucion de estrellas en Wise:')
print(df_wise['stars'].value_counts().sort_index())


---
## Paso 3: Limpieza de texto

Antes de aplicar los modelos de NLP, limpiamos las resenas:

1. **Combinamos `title` + `review`** en un solo campo `text_clean` para dar mas contexto al modelo.
2. Aplicamos la funcion `clean_text()` que:
   - Convierte a minusculas.
   - Elimina URLs.
   - Elimina emojis y caracteres no ASCII.
   - Elimina puntuacion (conservando letras, numeros y espacios).
   - Normaliza espacios multiples.

Primero inspeccionamos una muestra de resenas para entender la calidad del texto.


In [ ]:
# Muestra aleatoria de resenas sin limpiar
df_wise['review'].sample(5, random_state=42).tolist()


In [ ]:
def clean_text(text):
    """Limpieza basica de resenas en ingles."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)   # URLs
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)      # emojis / no ASCII
    text = re.sub(r'[^a-z0-9\s]', ' ', text)         # puntuacion
    text = re.sub(r'\s+', ' ', text).strip()          # espacios multiples
    return text

def combine_text(row):
    """Combina title + review para dar mas contexto al modelo."""
    t = str(row['title'])  if pd.notna(row.get('title'))  else ''
    r = str(row['review']) if pd.notna(row.get('review')) else ''
    return clean_text(t + ' ' + r)

# Aplicar limpieza
df_wise['text_clean']   = df_wise.apply(combine_text, axis=1)
df_sector['text_clean'] = df_sector.apply(combine_text, axis=1)

# Verificacion
n_empty = (df_wise['text_clean'] == '').sum()
print(f'Textos vacios en Wise: {n_empty}')
print()
print('Ejemplo de texto limpio:')
print(df_wise['text_clean'].iloc[0])


---
## Paso 4: Analisis de sentimiento con BERT

Utilizamos el modelo pre-entrenado **`nlptown/bert-base-multilingual-uncased-sentiment`**
de HuggingFace. Este modelo:

- Fue entrenado con resenas de productos en 6 idiomas.
- Devuelve una prediccion de **1 a 5 estrellas** por cada texto.
- Usamos la libreria `transformers` con un pipeline de `text-classification`.

Agrupamos las predicciones en 3 categorias:
- **1-2 estrellas** = Negativo
- **3 estrellas** = Neutro
- **4-5 estrellas** = Positivo

El modelo se aplica primero a las 100 resenas de Wise y despues a las ~5.700 del sector completo.
El texto se trunca a 512 tokens (limite del modelo BERT).


In [ ]:
# Cargar modelo de sentimiento (descarga ~700MB la primera vez)
sentiment_pipe = pipeline(
    'text-classification',
    model='nlptown/bert-base-multilingual-uncased-sentiment',
    truncation=True,
    max_length=512
)
print('Modelo de sentimiento cargado')


In [ ]:
def predict_sentiment(texts, pipe, batch_size=32):
    """Aplica el modelo de sentimiento a una lista de textos.
    Devuelve dos listas: estrellas predichas y etiquetas de sentimiento."""
    texts_trunc = [t[:512] if t else 'no review' for t in texts]
    preds = pipe(texts_trunc, batch_size=batch_size)
    stars_list, sent_list = [], []
    for p in preds:
        s = int(p['label'].split()[0])  # '4 stars' -> 4
        stars_list.append(s)
        sent_list.append(stars_to_label(s))
    return stars_list, sent_list

# Sentimiento Wise
print('Calculando sentimiento para Wise (100 resenas)...')
sp, sl = predict_sentiment(df_wise['text_clean'].tolist(), sentiment_pipe)
df_wise['stars_pred'] = sp
df_wise['sentiment']  = sl
print('Resultado:')
print(df_wise['sentiment'].value_counts())


In [ ]:
# Sentimiento para todo el sector
print('Calculando sentimiento para el sector completo (~5.700 resenas, puede tardar unos minutos)...')
sp2, sl2 = predict_sentiment(df_sector['text_clean'].tolist(), sentiment_pipe)
df_sector['stars_pred'] = sp2
df_sector['sentiment']  = sl2
print('Resultado del sector:')
print(df_sector['sentiment'].value_counts())


---
## Paso 5: Visualizacion del sentimiento

### 5.1 Sentimiento global de Wise
Grafico de barras con el porcentaje de resenas positivas, neutras y negativas de Wise.

### 5.2 Comparativa con la competencia
Grafico horizontal con el % de resenas positivas de **todas las empresas del sector**,
resaltando la posicion de Wise con color rojo y linea vertical.


In [ ]:
# 5.1 Sentimiento global de Wise
order = ['Positivo', 'Neutro', 'Negativo']
wise_counts = df_wise['sentiment'].value_counts().reindex(order, fill_value=0)
wise_pct    = (wise_counts / wise_counts.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(order, wise_pct.values,
              color=[PALETTE[o] for o in order], edgecolor='white', linewidth=1.5)
for bar, pct in zip(bars, wise_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pct}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_title('Sentimiento global - Wise.com', fontsize=14, fontweight='bold')
ax.set_ylabel('% resenas')
ax.set_ylim(0, 100)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
sns.despine()
plt.tight_layout()
plt.savefig('wise_sentimiento_global.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# 5.2 Comparativa con la competencia
sent_comp = (df_sector.groupby('company')['sentiment']
             .value_counts(normalize=True).mul(100).rename('pct').reset_index())
pos_comp  = sent_comp[sent_comp['sentiment'] == 'Positivo'].sort_values('pct', ascending=True)

colors = ['#e74c3c' if c == TARGET else '#3498db' for c in pos_comp['company']]

fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(pos_comp['company'], pos_comp['pct'], color=colors)
wise_val = pos_comp[pos_comp['company'] == TARGET]['pct'].values
if len(wise_val):
    ax.axvline(wise_val[0], color='#e74c3c', linestyle='--', linewidth=1.5, label='Wise')
ax.set_title('% Resenas Positivas - Sector Money & Insurance',
             fontsize=13, fontweight='bold')
ax.set_xlabel('% Positivo')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('wise_sentimiento_competencia.png', dpi=150, bbox_inches='tight')
plt.show()

# Posicion de Wise en el ranking
pos_comp_reset = pos_comp.reset_index(drop=True)
wise_idx = pos_comp_reset[pos_comp_reset['company'] == TARGET].index[0] + 1
print(f'Wise ocupa el puesto {wise_idx}/{len(pos_comp_reset)} en % positivo ({wise_val[0]:.1f}%)')
print(f'Media del sector: {pos_comp_reset["pct"].mean():.1f}%')


---
## Paso 6: Analisis de topics con BERTopic

**BERTopic** es un algoritmo de topic modeling que combina:
1. **Sentence-BERT** para generar embeddings semanticos de cada documento.
2. **UMAP** para reducir dimensionalidad.
3. **HDBSCAN** para clustering (agrupar documentos similares).
4. **c-TF-IDF** para extraer las palabras mas representativas de cada cluster/topic.

**Estrategia clave:** Entrenamos BERTopic sobre **todo el sector** (~5.700 resenas)
en vez de solo las 100 de Wise. Esto da al modelo suficientes documentos para
encontrar clusters robustos. Despues filtramos que topics aparecen en las resenas de Wise.

**Parametros elegidos:**
- `min_cluster_size=15`: clusters de al menos 15 documentos.
- `ngram_range=(1,2)`: permite bigramas (ej: 'customer service').
- `nr_topics='auto'`: BERTopic decide automaticamente cuantos topics mantener.


In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

docs = df_sector['text_clean'].tolist()

# Configuracion de componentes
umap_model    = UMAP(n_neighbors=10, n_components=5, min_dist=0.0, random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean', prediction_data=True)
vectorizer    = CountVectorizer(stop_words='english', ngram_range=(1, 2), min_df=3)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    nr_topics='auto',
    calculate_probabilities=True,
    verbose=True
)

print('Entrenando BERTopic sobre el sector completo...')
topics, probs = topic_model.fit_transform(docs)
df_sector['topic'] = topics

n_t = topic_model.get_topic_info()['Topic'].nunique()
print(f'Topics encontrados: {n_t}')
topic_model.get_topic_info().head(15)


In [ ]:
# Palabras clave de cada topic
topic_info = topic_model.get_topic_info()
for tid in sorted(topic_info['Topic'].values):
    if tid == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(tid)[:8]]
    sep = ' | '
    print(f'Topic {tid}: {sep.join(words)}')


### Etiquetado manual de topics

BERTopic asigna numeros a los topics. Revisamos las palabras clave de arriba
y asignamos nombres legibles. El topic -1 son outliers (documentos que no
encajan en ningun cluster).

**IMPORTANTE:** Los nombres de abajo deben ajustarse segun las palabras clave
que genere BERTopic en cada ejecucion, ya que pueden variar ligeramente.


In [ ]:
# Etiquetas de topics relevantes
TOPIC_LABELS = {
    -1: 'Sin topic (outlier)',
     0: 'Banca general / Fintech',
     2: 'Proceso facil y rapido',
     6: 'Atencion al cliente',
     7: 'Neobancos (Monzo/Monese)',
     9: 'Criptomonedas (Kraken)',
    13: 'Credito y prestamos (Zopa)',
    20: 'Banca para empresas (Tide)',
    32: 'Verificacion de identidad',
    34: 'Apertura de cuenta',
    35: 'Problemas de acceso / App',
}
df_sector['topic_label'] = df_sector['topic'].map(TOPIC_LABELS).fillna('Otros temas')
print('Labels asignados')


---
## Paso 7: Topics en Wise y comparativa con la competencia

Filtramos las resenas de Wise del dataframe del sector (que ya tiene topics asignados)
y analizamos:
- Que topics aparecen en las resenas de Wise.
- Como se distribuyen los topics en el resto del sector (heatmap).

**Hallazgo clave:** De las 100 resenas de Wise, 55 son outliers y 41 caen en
el topic generico 'Banca general / Fintech'. Solo 4 resenas caen en topics especificos.


In [ ]:
# Topics asignados a las resenas de Wise
df_wise_topics = df_sector[df_sector['company'] == TARGET].copy()
print('Distribucion de topics en Wise:')
print(df_wise_topics[['topic','topic_label']].value_counts().to_string())


In [ ]:
# Grafico: Topics en Wise (excluyendo outliers)
df_w_notout = df_wise_topics[df_wise_topics['topic'] != -1]
wise_topic_counts = df_w_notout['topic_label'].value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
wise_topic_counts.plot(kind='barh', ax=ax, color='#3498db', edgecolor='white')
ax.set_title('Topics en resenas de Wise.com', fontsize=13, fontweight='bold')
ax.set_xlabel('Numero de resenas')
sns.despine()
plt.tight_layout()
plt.savefig('wise_topics.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Heatmap: distribucion de topics por empresa en el sector
df_sect_notout = df_sector[df_sector['topic'] != -1]
topic_by_company = df_sect_notout.groupby(['company','topic_label']).size().reset_index(name='count')
pivot = topic_by_company.pivot_table(index='company', columns='topic_label',
                                      values='count', fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0).mul(100).round(1)

# Mostrar solo topics etiquetados
cols_show = [c for c in pivot_pct.columns if c != 'Otros temas']
pivot_show = pivot_pct[cols_show] if cols_show else pivot_pct

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(pivot_show, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': '% resenas'})
ax.set_title('Distribucion de Topics por empresa - Sector',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('wise_topics_competencia.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Paso 8: Sentimiento por topic en Wise

Cruzamos los dos analisis anteriores: para cada resena de Wise que tiene un topic
asignado, miramos que sentimiento tiene. Esto responde a la pregunta:
**En que temas la gente esta contenta y en cuales frustrada?**

Visualizamos con un grafico de barras apiladas (stacked bar).


In [ ]:
# Sentimiento por topic en Wise
wise_cross = (df_w_notout.groupby(['topic_label','sentiment'])
              .size().reset_index(name='count'))
pivot_wise = wise_cross.pivot_table(index='topic_label', columns='sentiment',
                                     values='count', fill_value=0)
for col in ['Positivo', 'Neutro', 'Negativo']:
    if col not in pivot_wise.columns:
        pivot_wise[col] = 0
pivot_wise = pivot_wise[['Positivo', 'Neutro', 'Negativo']]
pivot_wise_pct = pivot_wise.div(pivot_wise.sum(axis=1), axis=0).mul(100)

print('Sentimiento por topic en Wise (%):')
print(pivot_wise_pct.round(1))

fig, ax = plt.subplots(figsize=(9, 5))
pivot_wise_pct.plot(kind='barh', stacked=True, ax=ax,
                    color=[PALETTE['Positivo'], PALETTE['Neutro'], PALETTE['Negativo']],
                    edgecolor='white')
ax.set_title('Sentimiento por Topic - Wise.com', fontsize=13, fontweight='bold')
ax.set_xlabel('% resenas')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(title='Sentimiento', bbox_to_anchor=(1.02, 1), loc='upper left')
sns.despine()
plt.tight_layout()
plt.savefig('wise_sentimiento_por_topic.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Paso 9: Wise vs Competencia - Gap por topic

Calculamos el **gap** entre el % de resenas positivas de Wise y la media del sector
para cada topic. Un gap positivo (verde) indica que Wise supera al sector;
un gap negativo (rojo) indica un area de mejora.

**Nota importante:** Los topics donde Wise tiene 0 resenas (Kraken, Zopa, etc.)
generan gaps artificialmente grandes. Solo los topics donde Wise SI tiene resenas
son directamente accionables.


In [ ]:
def pct_positive(group):
    """Calcula el % de resenas positivas en un grupo."""
    return (group['sentiment'] == 'Positivo').mean() * 100

# % positivo por (empresa, topic)
sent_by_tc = (df_sect_notout.groupby(['company','topic_label'])
              .apply(pct_positive).reset_index(name='pct_positive'))

# Media del sector vs Wise
sector_avg = sent_by_tc.groupby('topic_label')['pct_positive'].mean().reset_index(name='sector_avg')
wise_avg   = (sent_by_tc[sent_by_tc['company'] == TARGET][['topic_label','pct_positive']]
              .rename(columns={'pct_positive':'wise_pct'}))
compare = sector_avg.merge(wise_avg, on='topic_label', how='left').fillna(0)
compare['gap'] = compare['wise_pct'] - compare['sector_avg']
compare = compare.sort_values('gap', ascending=True)

print('Gap Wise vs media del sector por topic:')
print(compare.to_string(index=False))


In [ ]:
# Grafico de gap
colors_gap = ['#2ecc71' if g >= 0 else '#e74c3c' for g in compare['gap']]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(compare['topic_label'], compare['gap'], color=colors_gap, edgecolor='white')
ax.axvline(0, color='black', linewidth=1.2)
ax.set_title('Wise vs Media del sector - Gap sentimiento positivo por topic',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Gap (puntos porcentuales)')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
sns.despine()
plt.tight_layout()
plt.savefig('wise_gap_competencia_por_topic.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Paso 10: Conclusiones y areas de mejora

### WordCloud de resenas negativas
Visualizamos las palabras mas frecuentes en las resenas negativas de Wise
para identificar patrones cualitativos adicionales.


In [ ]:
from wordcloud import WordCloud

neg_text = ' '.join(df_wise_topics[df_wise_topics['sentiment'] == 'Negativo']['text_clean'].tolist())
wc = WordCloud(width=800, height=400, background_color='white',
               colormap='Reds', max_words=80).generate(neg_text)

fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Palabras mas frecuentes en resenas NEGATIVAS - Wise.com',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('wise_wordcloud_negativo.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Resumen final
print('=' * 65)
print('CONCLUSIONES FINALES - WISE.COM')
print('=' * 65)

dist = df_wise_topics['sentiment'].value_counts(normalize=True).mul(100).round(1)
print()
print('1. SENTIMIENTO GLOBAL:')
for label in ['Positivo', 'Neutro', 'Negativo']:
    val = dist.get(label, 0)
    print(f'   {label}: {val}%')

# Ranking
pos_comp_r = pos_comp.reset_index(drop=True)
wise_rows = pos_comp_r[pos_comp_r['company'] == TARGET].index
if len(wise_rows):
    pos = wise_rows[0] + 1
    tot = len(pos_comp_r)
    pct_w = pos_comp_r.loc[wise_rows[0], 'pct']
    print(f'\n2. RANKING: Puesto {pos}/{tot} en % resenas positivas ({pct_w:.1f}%)')
    print(f'   Media del sector: {pos_comp_r["pct"].mean():.1f}%')

# Areas de mejora
weak = compare[compare['gap'] < 0].sort_values('gap')
# Filtrar solo topics donde Wise tiene resenas
wise_topic_ids = df_wise_topics[df_wise_topics['topic'] != -1]['topic_label'].unique()
weak_real = weak[weak['topic_label'].isin(wise_topic_ids)]

print('\n3. AREAS DE MEJORA (topics con resenas de Wise):')
for _, row in weak_real.iterrows():
    tl = row['topic_label']
    wp = row['wise_pct']
    sa = row['sector_avg']
    g  = row['gap']
    print(f'   [-] {tl}: Wise {wp:.1f}% vs Sector {sa:.1f}% (gap {g:.1f}pp)')

strong = compare[compare['gap'] > 0].sort_values('gap', ascending=False)
print('\n4. VENTAJAS COMPETITIVAS:')
for _, row in strong.iterrows():
    tl = row['topic_label']
    wp = row['wise_pct']
    sa = row['sector_avg']
    g  = row['gap']
    print(f'   [+] {tl}: Wise {wp:.1f}% vs Sector {sa:.1f}% (+{g:.1f}pp)')

print('\n' + '=' * 65)


---
## Resumen de conclusiones

| Metrica | Valor |
|---|---|
| Sentimiento dominante | **Negativo (55%)** |
| % Positivo | 34% |
| Ranking sector | Puesto 23/69 |
| Media sector (% positivo) | 45.9% |
| Topic principal | Banca general / Fintech (41 resenas) |
| Ventaja competitiva | Banca general: +13pp sobre la media |
| Areas de mejora | Atencion al cliente, Proceso, Apertura de cuenta |

### Recomendaciones para Wise
1. **Atencion al cliente**: Principal punto de dolor. Invertir en soporte mas rapido y humano.
2. **Proceso de verificacion/apertura**: Simplificar el onboarding y la verificacion KYC.
3. **Comunicacion proactiva**: Los clientes se quejan de falta de informacion durante
   bloqueos de cuenta y retrasos en transferencias.
4. **Capitalizar fortaleza en transferencias**: El servicio core de Wise (cambio de divisa
   y transferencias) tiene mejor percepcion que la media del sector.
